## Packages


In [ ]:
# Trading Simulation with AI Agents

## Setup and Dependencies
!pip install edsl  # Install the EDSL library needed for the agents


ERROR: Operation cancelled by user


## SET UP LLM AND API


In [ ]:
import os
LLM_MODEL = "gemini-2.0-flash-thinking-exp"

from google.colab import userdata
gemini_key = userdata.get('GOOGLE_API_KEY')
# EXPECTED_PARROT_API_KEY = userdata.get('EXPECTED_PARROT_API_KEY')

## Prompts

In [ ]:
# Create prompt directory and necessary prompt files
import os

# Create prompt directory if it doesn't exist
os.makedirs('./prompt', exist_ok=True)

# Create prompt directory if it doesn't exist
os.makedirs('./results', exist_ok=True)

In [ ]:
# Create prompt directory if it doesn't exist
os.makedirs('./prompt', exist_ok=True)

# Writing instruction.txt
instruction_text = """In this experiment, you will be playing a trading game with other participants. All of you will be given the same initial amount of 🔴 red chips, 🟢 green chips, 🔵 blue chips and 🟣 purple chips, but you may value the different colors of chips differently.
By making and accepting offers, you will try to exchange chips with the other players to increase the total value of chips that you end up holding at the end of the game.
You will receive payment based on the value of your chips.

You will play this trading game two times, against different groups of participants. In each game, you and the other participants will start with:
🔴 10 red chips
🟢 10 green chips
🔵 10 blue chips
🟣 10 purple chips

Valuations:
Each 🟢 green chip is worth $0.50 to each participant. However, you will all have different valuations for the red, blue, and purple chips, randomly chosen between $0.10 and $1.00. For example, Cat might value 🔴 red chips at $0.30 each, 🔵 blue chips at $0.70 each, and 🟣 purple chips at $0.50 each, while Mouse might value 🔴 red chips at $0.80 each, 🔵 blue chips at $0.30 each, and 🟣 purple chips at $0.60 each.
What this means:
Because each participant values the chips differently, there may be good reasons to trade. For instance, if you don’t care much about 🔴 red chips but someone else does, you might offer your red chips to them in exchange for 🟢 green chips, which you like more. Similarly, you might trade 🔵 blue chips with another participant if they value them more than you do. Additionally, you may want to exchange 🟣 purple chips if they hold different values for different participants. In this way, both you and the other participant can end up with chips that you find more valuable.
You know your own chip valuation and that everyone values 🟢 green chips the same, at $0.50 per chip. However, you do not know the other players' valuations for the other chips
To be clear, your ultimate goal is to make as much money as possible.
"""

with open('./prompt/instruction.txt', 'w') as f:
    f.write(instruction_text)


In [ ]:
# Writing rule.txt
rule_text = """How the game works:
The game consists of 3 rounds of trading. During each round, each player will have a turn to propose 1 trade. These turns are pre-determined in a random order and the order stays the same in each round.

Trade proposals:
To propose a trade, a player must:
1. Request a certain quantity of chips of a single color from any other player to get.
2. Specify a certain quantity of chips of a different color to give in return.

Trade rules:
Players cannot offer more chips than they currently hold. For example, if you only have 5 red chips, you cannot offer 6 red chips.
Players cannot trade chips of the same color. For example, you cannot trade red chips for red chips.

Trade completion:
When an offer is presented, all other active participants get a chance to accept or decline. Note: Active participants are those not currently making the offer.

Participants make their decisions simultaneously and privately. The participant who receives the offer is not dependent on who accepts the trade first. Some possible outcomes:
If no one accepts, the trade does not happen, and the turn ends.
If multiple participants accept, one accepting participant is chosen at random to complete the trade with the offering participant. This means that participants cannot choose who they trade with.
If only one participant accepts, the trade will happen.

Key points to remember:
In each round, each player gets to propose one trade and respond to other player's trades
You can only propose trades between different colored chips, and cannot offer to give a chip amount that you do not have
When multiple players accept a trade, the trading partner is randomly selected
"""

with open('./prompt/rule.txt', 'w') as f:
    f.write(rule_text)


In [ ]:
# Writing propose.txt
propose_text = """You are {{name}}.
Your valuations of the different types of chips are: {{preference_description}}.
You now have the following amounts of each chip: {{item}}.
The conversation history so far is {{history}}.

REMEMBER, to propose a trade, you must:
Request a certain quantity of chips of a single color from any other player to get
Specify a certain quantity of chips of a different color to give in return

REMEMBER you have the following amounts of each chip: {{item}}.
Your goal is to make as much money as possible. The trades, you choose to make to accomplish this, are up to you.
As a part of making money you uou must be rational - do not propose a trade in which you lose money. The value of a trade to you is the difference between the total value of chips you receive (quantity × your valuation) minus the total value of chips you give up (quantity × your valuation). Only propose trades that give you positive value.
In short, your trades should be both incentive compatible and incentive rational.
Your response must use these EXACT tags below. The response should include nothing else besides the tags, your trade offer, and your reasoning. The text between tags should be concise.
```
<REASONING>
[Provide your concise reasoning in a few sentences, e.g. To gain more surplus, I want more xxx chips]
</REASONING>

<CHECK>
[check if you have sufficient chips to trade. If you have n green chips, you can at most give n green chips. If you don't want to trade, you can ask for a large amount of chips that no one can afford]
<\CHECK>

<GET_COLOR> Color, e.g. red</GET_COLOR>
<GET_QUANTITY> quantity, e.g. n </GET_QUANTITY>
<GIVE_COLOR> Color, e.g. red</GIVE_COLOR>
<GIVE_QUANTITY> quantity, e.g. n </GIVE_QUANTITY>
```
"""

with open('./prompt/propose.txt', 'w') as f:
    f.write(propose_text)

In [ ]:
# Writing answer.txt
answer_text = """You are {{name}}.
Your valuations of the different types of chips are: {{preference_description}}.
You now have the following amounts of each chip: {{item}}.
The conversation history so far is {{history}}.

You have an offer. {{proposer}} is offering to give {{give}} and get {{get}} in return.
If you make this trade, your total wealth will change by: {{delta_surplus}}

Now, you need to decide whether to accept or decline.
Your response must use these EXACT tags below. The response should include nothing else besides the tags, your choice to accept or decline, and your reasoning. The text between tags should be concise.
```
<REASONING>
[Provide your concise reasoning in a few sentences.]
</REASONING>

<CHOICE>Yes or No </CHOICE>
```
"""

with open('./prompt/answer.txt', 'w') as f:
    f.write(answer_text)

## Code for LLM simulation

In [ ]:
## Imports and Data Classes
from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple, List, Any
import re
from datetime import datetime
from edsl import Model, Agent
from edsl.questions import QuestionFreeText, QuestionYesNo
from edsl.prompts import Prompt
from edsl import Agent, Scenario, Survey
from itertools import cycle, combinations
import random
import csv
import json
from dataclasses import fields
import os
import concurrent.futures
import sys

ModuleNotFoundError: No module named 'edsl'

In [ ]:
import base64
import os
import time
from google import genai
from google.genai import types

def generate(prompt):
    client = genai.Client(
        api_key=gemini_key
    )

    model = LLM_MODEL
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text=prompt
                ),
            ],
        ),
    ]
    generate_content_config = types.GenerateContentConfig(
        temperature=0.7,
        top_p=0.95,
        top_k=64,
        max_output_tokens=65536,
        response_mime_type="text/plain",
    )

    def get_response():
      response = ''
      for chunk in client.models.generate_content_stream(
          model=model,
          contents=contents,
          config=generate_content_config,
      ):
          response += chunk.text
      return response

    for i in range(5):
      try:
        return get_response()
      except:
        print("retrying")
        time.sleep(10)
        continue
    raise Exception("Failed to generate response")


generate("hi")

In [ ]:
@dataclass
class TradeProposal:
    # Core trade info
    turn_id: int
    round_id: int
    proposal_id: int
    proposer_name: str
    receiver_name: str
    chips_offered: Dict[str, int]
    chips_requested: Dict[str, int]
    timestamp: datetime = field(default_factory=datetime.now)
    status: str = "PENDING"  # e.g., 'PENDING', 'ACCEPTED', 'REJECTED'
    rationale: str = ""      # brief explanation

    ## player info
    player1 : str = ""
    player2 : str = ""
    player3 : str = ""

    # Action data: who is sender, who is chosen receiver, etc.
    player1_is_sender: bool = False
    player2_is_sender: bool = False
    player3_is_sender: bool = False
    player1_is_chosen_receiver: bool = False
    player2_is_chosen_receiver: bool = False
    player3_is_chosen_receiver: bool = False

    player1_offer_response: str = ""
    player2_offer_response: str = ""
    player3_offer_response: str = ""

    buy_offer_red:  int = 0
    buy_offer_green: int = 0
    buy_offer_blue: int = 0
    buy_offer_purple: int = 0
    sell_offer_red:  int = 0
    sell_offer_green:  int = 0
    sell_offer_blue:  int = 0
    sell_offer_purple:  int = 0

    # State data: chip counts before/after for each player & chip
    player1_red_before: int = 0
    player1_red_after: int = 0
    player2_red_before: int = 0
    player2_red_after: int = 0
    player3_red_before: int = 0
    player3_red_after: int = 0

    player1_green_before: int = 0
    player1_green_after: int = 0
    player2_green_before: int = 0
    player2_green_after: int = 0
    player3_green_before: int = 0
    player3_green_after: int = 0

    player1_blue_before: int = 0
    player1_blue_after: int = 0
    player2_blue_before: int = 0
    player2_blue_after: int = 0
    player3_blue_before: int = 0
    player3_blue_after: int = 0

    player1_purple_before: int = 0
    player1_purple_after: int = 0
    player2_purple_before: int = 0
    player2_purple_after: int = 0
    player3_purple_before: int = 0
    player3_purple_after: int = 0

    player1_red_value: int= 0
    player1_blue_value: int= 0
    player1_green_value: int= 0
    player1_purple_value: int= 0

    player2_red_value: int= 0
    player2_blue_value: int= 0
    player2_green_value: int= 0
    player2_purple_value: int= 0

    player3_red_value: int= 0
    player3_blue_value: int= 0
    player3_green_value: int= 0
    player3_purple_value: int= 0

    # Surplus / utility data: payouts (or utilities) before & after
    player1_payout_before: float = 0.0
    player1_payout_after: float = 0.0
    player2_payout_before: float = 0.0
    player2_payout_after: float = 0.0
    player3_payout_before: float = 0.0
    player3_payout_after: float = 0.0

    sender_payout_before: float = 0.0
    sender_payout_after: float = 0.0
    recipient_payout_before: float = 0.0
    recipient_payout_after: float = 0.0

    do_proposer_gain: bool=False
    proposal_content: str = ""


def convert_dict_lang(dict_str):
    result = []
    for key, value in dict_str.items():
        result.append(f"{value} {key} chips")
    return ", ".join(result)

def convert_prompt_str(dict_str):
    ## get rid of the {}
    return dict_str.replace("{", "(").replace("}", ")")

class Player:
    def __init__(self, name: str, valuation:Dict):
        self.name = name
        self.orig_item = {
            "green": 10,
            "red": 10,
            "blue": 10,
            "purple": 10
        }
        self.item = dict(self.orig_item)
        self.valuation = valuation

        self.model = Model(LLM_MODEL)
        self.cache = None  # or pass an existing Cache in constructor

        # Agent traits example
        self.instruction = Prompt.from_txt('./prompt/instruction.txt')
        self.rule = Prompt.from_txt('./prompt/rule.txt')


    def reinitiate(self):
        self.item = dict(self.orig_item)

    def propose_trade(
        self,
        history: str = "",
        system_warning: str ="",
    ) -> str:
        """
        Use LLM to propose a trade. Return the raw text for further parsing.
        """
        # s = Scenario({"instruction": self.instruction, "rule": self.rule})
        propose_prompt_text = Prompt.from_txt("./prompt/propose.txt")
        propose_prompt = propose_prompt_text.render(
            {
            "name": self.name,
            "preference_description":convert_prompt_str(str(self.valuation)),
            "item": convert_dict_lang(self.item),
            "history": convert_prompt_str(str(history)),
            })
        print(str(propose_prompt))

        prompt = str(self.instruction)+ str(self.rule) +str(propose_prompt) + system_warning

        #q = QuestionFreeText(
        #    question_text = prompt,
        #    question_name = "propose"
        #)

        #results= q.by(self.model).run()
        #text_proposal = results.select("propose").to_list()[0]
        # print("===========", self.name ,'\n',text_proposal)

        text_proposal = generate(prompt)

        return text_proposal

    def answer_trade(self, get_str: Dict, give_str: Dict, other_agent_name: str, history: str, delta_surplus: float) -> str:
        """
        Use LLM to decide whether to accept a trade. Return 'Yes' or 'No'.
        """
        prompt_answer_text = Prompt.from_txt("./prompt/answer.txt")

        prompt_answer = prompt_answer_text.render(
            {
                "name": self.name,
                "preference_description": convert_prompt_str(str(self.valuation)),
                "item": convert_dict_lang(self.item),
                "history": convert_prompt_str(str(history)),
                "proposer":other_agent_name,
                "give": convert_dict_lang(give_str),
                "get": convert_dict_lang(get_str),
                "delta_surplus":delta_surplus
            })

        # print(str(prompt_answer))
        question_text = str(self.instruction)+ str(self.rule) + str(prompt_answer)
        #q = QuestionFreeText(
        #    question_text = str(self.instruction)+ str(self.rule) + str(prompt_answer),
        #    question_name = "answer"
        #)
        #results= q.by(self.model).run()
        #answer = results.select("answer").to_list()[0]
        answer = generate(question_text)
        return answer


    def calculate_utility(self, items: dict, valuations: dict) -> float:
        """Utility = sum of (chip_count * chip_valuation)."""
        return sum(items.get(chip, 0) * valuations.get(chip, 0.0)
                   for chip in items.keys())

    def has_sufficient_chips(self, chips_needed: dict) -> bool:
        for chip, qty in chips_needed.items():
            if self.item.get(chip, 0) < qty:
                return False
        return True

    def execute_trade(self,
                      chips_offered: Dict[str, int],
                      chips_requested: Dict[str, int],
                      other_player: 'Player'):
        """
        Adjust inventories for both parties if the trade is accepted.
        """
        # self loses chips_offered, gains chips_requested
        for chip, amt in chips_offered.items():
            self.item[chip] -= amt
            other_player.item[chip] += amt
            #   Check if any item becomes negative
            if self.item[chip] < 0:
                raise ValueError(f"Insufficient {chip} for self after transaction")

        # other_player loses chips_requested, gains chips_offered
        for chip, amt in chips_requested.items():
            other_player.item[chip] -= amt
            self.item[chip] += amt

            # Check if any item becomes negative for other_player
            if other_player.item[chip] < 0:
                raise ValueError(f"Insufficient {chip} for other player after transaction")

In [ ]:
class Negotiation:
    def __init__(self, players: List[Player], valuations: Dict[str, Dict[str, float]]):
        """
        players: list of Player objects
        valuations: e.g. {'Alice': {'green': 1.0, 'red': 3.0}, ...}
        """
        self.players = players
        self.valuations = valuations
        self.trade_history: List[TradeProposal] = []
        self.proposal_counter = 0
        self.memory = []

    def reinitiate_all(self):
        for p in self.players:
            p.reinitiate()
        self.trade_history.clear()
        self.proposal_counter = 0


    def scenario4_llm_proposals_llm_accepts(self, sequence):
        """
        Scenario 4:
        - A single player (the proposer) uses LLM to propose a trade.
        - We parse that proposal to get (chips_offered, chips_requested).
        - multiple other player(s) uses LLM to decide to accept or reject.
        """
        round_id = 0
        for counting, i1 in enumerate(sequence):
            if counting % 3 == 0 and counting != 0:  # Increment round_id every 3 iterations (after the first three)
                round_id += 1
            proposer = self.players[i1]
            # print("--------------", proposer.name)

            # 1) Capture pre-trade states
            before_items = {p.name: dict(p.item) for p in self.players}
            before_utilities = {
                p.name: p.calculate_utility(p.item, self.valuations[p.name])
                for p in self.players
            }

            # Step 2) LLM propose
            system_warning = ""
            for attempt in range(5):  # up to 5 tries to find a beneficial proposal
                try:
                    raw_text = proposer.propose_trade(history=self.memory, system_warning=system_warning)
                    # parse the raw_text into structured data
                    print(raw_text)
                    chips_offered, chips_requested = self.parse_proposal_text(raw_text)
                    if not chips_offered or not chips_requested:
                        system_warning = f"Your previoous proposa is invalid. Try again."
                        continue
                    # 2) Check if proposer has enough chips to offer
                    elif not self.has_sufficient_chips_to_offer(proposer, chips_offered):
                        system_warning = f"Your previoous proposal: {convert_dict_lang(chips_offered)} will lead to Insufficient chip to trade. Try again."
                        continue
                    else:
                        ## check the chips_offered is of the right format
                        _ = convert_dict_lang(chips_offered)
                        _ = convert_dict_lang(chips_requested)
                        break

                except (ValueError, IndexError) as e:
                    print(f"Attempt {attempt + 1} failed: {e}")
                    if attempt == 4:
                        raise
                    continue

            # Step 3: Among the other players, see who is willing to accept
            possible_acceptors = []
            accepted = False
            print(chips_offered, chips_requested)
            do_proposer_gain = self.does_proposer_gain(
                    proposer=proposer,
                    chips_offered= chips_offered,
                    chips_requested= chips_requested,
                    proposer_val =self.valuations[proposer.name]
            )
            for p1 in self.players:
                # Skip if p1 is the propose

                if p1 != proposer:
                    # Check if player has sufficient chips
                    if p1.has_sufficient_chips(chips_requested):
                        # Calculate utility improvement for the receiver
                        delta_surplus = self.cal_receiver_gain(
                            receiver=p1,
                            chips_offered=chips_offered,
                            chips_requested=chips_requested,
                            receiver_val=self.valuations[p1.name]
                        )

                        # Query the LLM
                        answer_text = p1.answer_trade(
                            give_str=chips_offered,
                            get_str=chips_requested,
                            other_agent_name=proposer.name,
                            history=self.memory,
                            delta_surplus=delta_surplus
                        )
                        print("+++++++++++++++++", answer_text)

                        # Parse and evaluate the answer
                        answer = self.parse_answer_text(answer_text)
                        accepted = answer.strip().lower() in ["yes", "y", "accept"]
                        if accepted:
                            possible_acceptors.append(p1)


            # 4) If multiple accept, pick one randomly
            if len(possible_acceptors) > 1:
                receiver = random.choice(possible_acceptors)
                rationale = "Multiple acceptors"
            elif len(possible_acceptors) == 1:
                receiver = possible_acceptors[0]
                rationale = "Single acceptor"
            else:
                rationale = "No one accepted"

            if len(possible_acceptors)> 0:
                proposer.execute_trade(chips_offered, chips_requested, receiver)

            # 5) Execute trade
            after_items = {p.name: dict(p.item) for p in self.players}
            after_utilities = {
                p.name: p.calculate_utility(p.item, self.valuations[p.name])
                for p in self.players
            }

            # 6) Record
            self.record_trade(
                turn_id=i1,
                round_id = round_id,
                proposer_name=proposer.name,
                receiver_name=receiver.name if accepted else "No one",
                chips_offered=chips_offered,
                chips_requested=chips_requested,
                status="ACCEPTED" if accepted else "REJECTED",
                rationale=rationale,
                possible_acceptors=possible_acceptors,
                before_items=before_items,
                after_items=after_items,
                before_utilities=before_utilities,
                after_utilities=after_utilities,
                proposal_content=raw_text,
                do_proposer_gain=do_proposer_gain
            )

            self.update_memory(
                turn_id=i1,
                proposer_name=proposer.name,
                chips_offered=chips_offered,
                chips_requested=chips_requested,
                receiver_name=receiver.name if accepted else "No one",
            )

    def has_sufficient_chips_to_offer(self, player: Player, chips_offered: dict) -> bool:
        """
        Checks if `player` has enough chips to fulfill the proposed `chips_offered`.
        Returns True if the player's current items can cover all amounts in chips_offered,
        otherwise False.
        """
        for chip, amount in chips_offered.items():
            if player.item.get(chip, 0) < amount:
                return False
        return True

    def does_proposer_gain(self,
                           proposer: Player,
                           chips_offered: dict,
                           chips_requested: dict,
                           proposer_val: dict) -> bool:
        """
        Return True if the proposer’s utility increases by making this trade.
        """
        current_utility = proposer.calculate_utility(proposer.item, proposer_val)
        hypothetical = dict(proposer.item)

        # Subtract offered, add requested
        for chip, amt in chips_offered.items():
            if hypothetical[chip] < amt:
                # If we don't have enough to offer, can't do it
                return False
            hypothetical[chip] -= amt

        for chip, amt in chips_requested.items():
            hypothetical[chip] += amt

        new_utility = proposer.calculate_utility(hypothetical, proposer_val)
        return new_utility >= current_utility

    def cal_receiver_gain(self,
                           receiver: Player,
                           chips_offered: dict,
                           chips_requested: dict,
                           receiver_val: dict) -> bool:
        """
        calculate receiver’s utility change if they accept the trade.
        """
        # Current
        current_utility = receiver.calculate_utility(receiver.item, receiver_val)

        hypothetical = dict(receiver.item)
        # Gains chips_offered, loses chips_requested
        for chip, amt in chips_offered.items():
            hypothetical[chip] += amt
        for chip, amt in chips_requested.items():
            hypothetical[chip] -= amt

        new_utility = receiver.calculate_utility(hypothetical, receiver_val)
        return new_utility - current_utility


    def parse_proposal_text(self, text: str) -> Tuple[Dict[str, int], Dict[str, int]]:
        """
        Convert the raw text to structured data: (chips_offered, chips_requested).
        Handles input like:
        <GET_COLOR> red </GET_COLOR>
        <GET_QUANTITY> 2 </GET_QUANTITY>
        <GIVE_COLOR> blue </GIVE_COLOR>
        <GIVE_QUANTITY> 10 </GIVE_QUANTITY>
        """
        chips_offered = {}
        chips_requested = {}

        # Regex patterns to extract content from the tags
        get_color_pattern = r"<GET_COLOR>\s*(.*?)\s*</GET_COLOR>"
        get_quantity_pattern = r"<GET_QUANTITY>\s*(\d+)\s*</GET_QUANTITY>"
        give_color_pattern = r"<GIVE_COLOR>\s*(.*?)\s*</GIVE_COLOR>"
        give_quantity_pattern = r"<GIVE_QUANTITY>\s*(\d+)\s*</GIVE_QUANTITY>"

        # Find all GET and GIVE color and quantity values
        get_colors = re.findall(get_color_pattern, text, flags=re.IGNORECASE)
        get_quantities = re.findall(get_quantity_pattern, text, flags=re.IGNORECASE)
        give_colors = re.findall(give_color_pattern, text, flags=re.IGNORECASE)
        give_quantities = re.findall(give_quantity_pattern, text, flags=re.IGNORECASE)

        # Ensure there is a 1-to-1 match between colors and quantities
        if len(get_colors) == len(get_quantities):
            for color, quantity in zip(get_colors, get_quantities):
                try:
                    quantity = int(quantity.strip())
                    chips_requested[color.strip().lower()] = chips_requested.get(color.strip(), 0) + quantity
                except ValueError:
                    # Skip invalid quantity entries
                    continue

        if len(give_colors) == len(give_quantities):
            for color, quantity in zip(give_colors, give_quantities):
                try:
                    quantity = int(quantity.strip())
                    chips_offered[color.strip().lower()] = chips_offered.get(color.strip(), 0) + quantity
                except ValueError:
                    # Skip invalid quantity entries
                    continue

        return chips_offered, chips_requested

    def parse_answer_text(self, text: str):
        ## example input
        ## <CHOICE>Yes </CHOICE>
        pattern = r"<CHOICE>(.*?)</CHOICE>"
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            # Return the raw string within the <CHOICE> tag (stripped of whitespace).
            return match.group(1).strip()
        return ""


    def update_memory(self,
                    turn_id: int,
                    proposer_name: str,
                    receiver_name: str,
                    chips_offered: dict,
                    chips_requested: dict):

        give_chips = convert_dict_lang(chips_offered)
        get_chips = convert_dict_lang(chips_requested)
        proposal = f"In round {len(self.memory)}, {proposer_name} offered {give_chips} to get {get_chips}. "
        if receiver_name== "No one":
            response =  f"No deal: No one accepted {proposer_name}'s offer. \n"
        else:
            response = f"Deal made: {proposer_name}'s offer was accepted by {receiver_name}. \n"

        self.memory.append(proposal+response)

    def record_trade(self,
                    turn_id: int,
                    round_id: int,
                    proposer_name: str,
                    receiver_name: str,
                    chips_offered: dict,
                    chips_requested: dict,
                    status: str,
                    rationale: str,
                    possible_acceptors: list,
                    before_items: Dict[str, Dict[str, int]] = None,
                    after_items: Dict[str, Dict[str, int]] = None,
                    before_utilities: Dict[str, float] = None,
                    after_utilities: Dict[str, float] = None,
                    proposal_content: str="",
                    do_proposer_gain: bool=False
                    ):

        # Convenient lookups for each player's name
        p1_name = self.players[0].name
        p2_name = self.players[1].name
        p3_name = self.players[2].name

        # Utility lookups: if dict is None, default them
        if before_items is None:
            before_items = {p.name: dict(p.item) for p in self.players}
        if after_items is None:
            after_items = {p.name: dict(p.item) for p in self.players}
        if before_utilities is None:
            before_utilities = {
                p.name: p.calculate_utility(p.item, self.valuations[p.name])
                for p in self.players
            }
        if after_utilities is None:
            after_utilities = {
                p.name: p.calculate_utility(p.item, self.valuations[p.name])
                for p in self.players
            }

        # Identify who is sender, who is chosen recipient, etc.
        player1_is_sender = (proposer_name == p1_name)
        player2_is_sender = (proposer_name == p2_name)
        player3_is_sender = (proposer_name == p3_name)

        player1_is_selected_receiver = (receiver_name == p1_name)
        player2_is_selected_receiver = (receiver_name == p2_name)
        player3_is_selected_receiver = (receiver_name == p3_name)

        # Build the proposal object
        proposal = TradeProposal(
            turn_id = turn_id,
            round_id = round_id,
            proposal_id = self.proposal_counter,
            proposer_name = proposer_name,
            receiver_name = receiver_name,
            chips_offered = chips_offered,
            chips_requested = chips_requested,
            status = status,
            rationale = rationale,

            player1 = p1_name,
            player2 = p2_name,
            player3 = p3_name,

            # Mark which player is sender / receiver
            player1_is_sender = player1_is_sender,
            player2_is_sender = player2_is_sender,
            player3_is_sender = player3_is_sender,
            player1_is_chosen_receiver = player1_is_selected_receiver,
            player2_is_chosen_receiver = player2_is_selected_receiver,
            player3_is_chosen_receiver = player3_is_selected_receiver,

            player1_offer_response = True if self.players[0].name in possible_acceptors else False,
            player2_offer_response = True if self.players[1].name in possible_acceptors else False,
            player3_offer_response = True if self.players[2].name in possible_acceptors else False,

            # Optionally set buy/sell flags. For instance,
            # "Buy offer (RED)" means the proposer is requesting red chips.
            buy_offer_red = chips_requested["red"] if "red" in chips_requested else 0,
            buy_offer_green = chips_requested["green"] if "green" in chips_requested else 0,
            buy_offer_blue= chips_requested["blue"] if "blue" in chips_requested else 0,
            buy_offer_purple = chips_requested["purple"] if "purple" in chips_requested else 0,
            sell_offer_red = chips_offered["red"] if "red" in chips_offered else 0,
            sell_offer_green = chips_offered["green"] if "green" in chips_offered else 0,
            sell_offer_blue = chips_offered["blue"] if "blue" in chips_offered else 0,
            sell_offer_purple = chips_offered["purple"] if "purple" in chips_offered else 0,

            # Before/after states for each player & chip
            player1_red_before = before_items[p1_name]["red"],
            player1_red_after  = after_items[p1_name]["red"],
            player2_red_before = before_items[p2_name]["red"],
            player2_red_after  = after_items[p2_name]["red"],
            player3_red_before = before_items[p3_name]["red"],
            player3_red_after  = after_items[p3_name]["red"],

            player1_green_before = before_items[p1_name]["green"],
            player1_green_after  = after_items[p1_name]["green"],
            player2_green_before = before_items[p2_name]["green"],
            player2_green_after  = after_items[p2_name]["green"],
            player3_green_before = before_items[p3_name]["green"],
            player3_green_after  = after_items[p3_name]["green"],

            player1_blue_before = before_items[p1_name]["blue"],
            player1_blue_after  = after_items[p1_name]["blue"],
            player2_blue_before= before_items[p2_name]["blue"],
            player2_blue_after = after_items[p2_name]["blue"],
            player3_blue_before = before_items[p3_name]["blue"],
            player3_blue_after = after_items[p3_name]["blue"],

            player1_purple_before= before_items[p1_name]["purple"],
            player1_purple_after = after_items[p1_name]["purple"],
            player2_purple_before = before_items[p2_name]["purple"],
            player2_purple_after = after_items[p2_name]["purple"],
            player3_purple_before = before_items[p3_name]["purple"],
            player3_purple_after = after_items[p3_name]["purple"],

            # Valuations
            player1_red_value = self.valuations[p1_name]["red"],
            player1_blue_value= self.valuations[p1_name]["blue"],
            player1_green_value = self.valuations[p1_name]["green"],
            player1_purple_value = self.valuations[p1_name]["purple"],

            player2_red_value = self.valuations[p2_name]["red"],
            player2_blue_value = self.valuations[p2_name]["blue"],
            player2_green_value = self.valuations[p2_name]["green"],
            player2_purple_value = self.valuations[p2_name]["purple"],

            player3_red_value = self.valuations[p3_name]["red"],
            player3_blue_value = self.valuations[p3_name]["blue"],
            player3_purple_value = self.valuations[p3_name]["purple"],
            player3_green_value = self.valuations[p3_name]["green"],

            # Surplus / payout data
            player1_payout_before = before_utilities[p1_name],
            player1_payout_after  = after_utilities[p1_name],
            player2_payout_before = before_utilities[p2_name],
            player2_payout_after  = after_utilities[p2_name],
            player3_payout_before = before_utilities[p3_name],
            player3_payout_after  = after_utilities[p3_name],

            sender_payout_before = before_utilities[proposer_name],
            sender_payout_after = after_utilities[proposer_name],
            recipient_payout_before = before_utilities[receiver_name] if status == "ACCEPTED" else None,
            recipient_payout_after = after_utilities[receiver_name] if status == "ACCEPTED" else None,

            do_proposer_gain = do_proposer_gain,
            proposal_content = proposal_content
        )

        # Increment ID counter and save
        self.proposal_counter += 1
        self.trade_history.append(proposal)

    def convert_trade_history_to_rows(self, cohort_name, stage_name, mode) -> List[Dict]:
        """
        Convert self.trade_history (list of TradeProposal)
        to a list of dictionaries for CSV writing.
        """
        rows = []

        original_field_names = [f.name for f in fields(TradeProposal)]  # from dataclasses import fields
        field_names = ['mode','cohort_name', 'stage_name'] + original_field_names  # Place new fields first

        for proposal in self.trade_history:
            # Convert the TradeProposal dataclass to a dict
            row_dict = {
            field_name: getattr(proposal, field_name)
                for field_name in field_names if field_name not in ['mode','cohort_name', 'stage_name']
            }
            # Add cohort_name and stage_name to the row
            row_dict['mode'] = mode
            row_dict['cohort_name'] = cohort_name
            row_dict['stage_name'] = stage_name

            rows.append(row_dict)
        return rows

    def save_trade_history_to_csv(self, csv_path: str, cohort_name: str, stage_name: str, mode: str):
        """
        Write all fields from each TradeProposal in self.trade_history to CSV.
        """
        original_field_names = [f.name for f in fields(TradeProposal)]  # from dataclasses import fields
        field_names = ['mode','cohort_name', 'stage_name'] + original_field_names  # Place new fields first


        # Check if the file already exists
        file_exists = os.path.exists(csv_path)

        with open(csv_path, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=field_names)

            # Only write the header if the file doesn't exist
            if not file_exists:
                writer.writeheader()

            for proposal in self.trade_history:
                # Convert the TradeProposal dataclass to a dict
                row_dict = {
                field_name: getattr(proposal, field_name)
                    for field_name in field_names if field_name not in ['mode','cohort_name', 'stage_name']
                }
                # Add cohort_name and stage_name to the row
                row_dict['mode'] = mode
                row_dict['cohort_name'] = cohort_name
                row_dict['stage_name'] = stage_name
                writer.writerow(row_dict)

**Helper function to save data to CSV**

In [ ]:
def process_negotiations(json_file_path: str, output_csv_path: str, seed=42):
    """
    Reads a JSON file containing negotiation game data, extracts valuations,
    runs the negotiation ordering analysis, and writes results to a CSV.
    """
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    # Iterate over all games in the JSONx
    for game in data["games"]:
        # 1. Extract the cohort name
        cohort_name = game.get("cohortName", "unknown_cohort")
        stage_name = game.get("stageName", "unknown_stage")

        # 2. Extract participantChipValueMap and players
        metadata = game["data"]["metadata"]
        participant_map = metadata["participantChipValueMap"]
        players_name = metadata["players"]

        ## focus on the case where we have 3 players
        if len(players_name)<=2:
            continue

        print("=================================")
        print('players',cohort_name, stage_name)
        # 3. Build valuations in the order of `players`, for goods = [red, green, blue]
        valuations = {}
        for p in players_name:
            color_map = participant_map[p]
            # Force the goods order: [red, green, blue]
            valuations[p] = {
                                "red": round(color_map["RED"], 2),
                                "green": round(color_map["GREEN"], 2),
                                "blue": round(color_map["BLUE"], 2),
                                "purple": round(color_map["PURPLE"], 2)
                             }

        sequence = [0,1,2,0,1,2,0,1,2] ## Fixed order for the proposer
        #4. Build the negotiation environment

        # Create some players
        players = []
        for player in players_name:
            players.append(Player(name=player, valuation=valuations[player]))

        negotiation = Negotiation(players, valuations)

        # Scenario 4
        mode = 'llm'
        negotiation.scenario4_llm_proposals_llm_accepts(sequence=sequence)
        # print("Scenario 1 Trade History:")
        # for h in negotiation.trade_history:
        #     print(h)
        negotiation.save_trade_history_to_csv(output_csv_path, cohort_name=cohort_name, stage_name=stage_name, mode=mode)

        # JWEXLER: add break to only do one game
        # break


## Main function

In [ ]:
# Things to do before running main code
# 1. Upload the human experiment json to the results folder
# 2. create your LLM API key

In [ ]:
json_file = "/content/results/real_exp_4.json"
output_csv = "/content/results/llm_simulation.csv"  # Desired output CSV path

process_negotiations(json_file, output_csv)